In [1]:
from mltau.tools.evaluation import decode_ParTauDETR as dp
from mltau.tools.evaluation import set_to_set_models as sp
import awkward as ak

In [2]:
from hydra import compose, initialize
from omegaconf import OmegaConf

with initialize(version_base=None, config_path="../config", job_name="test_app"):
    cfg = compose(config_name="main_ParTauDETR")



In [3]:
from importlib import reload
reload(dp)
reload(sp)

<module 'mltau.tools.evaluation.set_to_set_models' from '/home/karl/ml-tau-model/mltau/tools/evaluation/set_to_set_models.py'>

In [4]:
import glob
import os.path

run_basename = '20260922_ParTauDETR_120epochs'
checkpoint_path = os.path.expanduser(f"~/runs/{run_basename}/models/ParTauDETR-model_best.ckpt")
data_paths = sorted(glob.glob("/scratch/persistent/karl/ml-tau/0921_ParTauDETR_dataset/z_test_*.parquet"))
assert(data_paths)

In [5]:
outputs, targets, _weights, reco_jet_p4s, _ = dp.model_inference(checkpoint_path, data_paths, cfg)

Read 590,793 jets from 6 parquet files.


In [6]:
data = ak.concatenate([ak.from_parquet(path) for path in data_paths])

In [7]:
obj_cls_trsh = 0.5
true_daughters, pred_daughters = dp.create_predictions(
    outputs, targets, _weights, reco_jet_p4s, cfg, obj_cls_trsh=obj_cls_trsh,
    # Objectness is trained on signal jets only; whether a jet is a tau is the
    # tauID head's call. On a sample with background jets set this to
    # cfg.model.detr.inference.tau_id_threshold; on z_test alone leave None.
    tau_threshold=None,
)

In [8]:
data_to_save = sp.construct_prediction_file_content(
    data, pred_daughters, true_daughters, cfg.dataset.tau_daughter_pdg_ids, debug = True,
)

In [9]:
obj_cls_trsh_str = f'{obj_cls_trsh:.3f}'.replace('.', 'p')
output_path = os.path.expanduser(f'~/tmp/{run_basename}-model_best-{obj_cls_trsh_str}thr.parquet')
output_dir = os.path.dirname(output_path)

import os
if not os.path.isdir(output_dir):
  os.makedirs(output_dir)

In [10]:
# dp.save_results(true_daughters, pred_daughters, output_path)
ak.to_parquet(data_to_save, output_path)

  created_by: parquet-cpp-arrow version 18.1.0
  num_columns: 46
  num_rows: 590793
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 0

In [11]:
data_to_save.fields

['reco_jet_p4',
 'gen_jet_p4',
 'reco_cand_p4s',
 'reco_cand_pdgs',
 'reco_cand_charges',
 'gen_jet_tau_vis_energy',
 'gen_jet_tau_decaymode',
 'gen_jet_tau_charge',
 'gen_jet_tau_full_p4',
 'gen_jet_tau_vis_daughter_p4s',
 'gen_jet_tau_vis_daughter_pdgs',
 'gen_jet_tau_vis_daughter_charges',
 'gen_jet_tau_p4',
 'file_id',
 'event_id',
 'pred_tau_daughter_meson_classes',
 'pred_tau_daughter_p4s',
 'pred_tau_daughter_charges',
 'tau_decaymode',
 'tau_p4',
 'tau_charge',
 'gen_jet_tau_decaymode_exp']